# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.** This notebook pulls together `w04`
(baseline), `w05` (model), `w06` (validation audit), and `w07` (action playbook) into the same
shape as the deployed paper at `docs/index.html` (URL recorded in `submission/paper_url.txt`).

## 1. Question

**Research question:** Given a page's trailing-90-day search and engagement history, which pages
should a content editor put at the top of this week's refresh queue — and does a learned model
beat a transparent, hand-written rule at answering that?

**Decision it supports:** which handful of pages, out of hundreds, an editor with limited hours
opens first. Right now that choice is gut feel or a simple sort; this project turns it into a
ranked, evidence-backed queue with a reason attached to every row.

**Who acts:** a content editor or SEO lead who reviews the top of the queue and decides to
refresh, rewrite metadata, expand thin sections, or leave a page alone — never an automated edit.

## 2. Data

**Source:** `data/raw/content_refresh_anonymized.csv` — the FlyRank ML Internship starter
release, 30,000 rows × 44 columns, one row per pseudonymized content item, 32 pseudonymized
clients, all metrics aggregated over a trailing 90-day window ending at export time (see
`docs/data-dictionary.md`).

**Why the starter slice, not the full ~79M-row warehouse:** the full warehouse is hosted on
Hugging Face behind a gated, token-based access flow designed for interactive notebook sessions
(Colab Secrets). The environment this capstone was finished in cannot reach that host or hold a
long-lived token, so every number in this capstone is computed on the public starter CSV. The
`w03_data_contract_(2).ipynb` exploration earlier in this repo shows the warehouse-scale workflow
(DuckDB over `hf://`) working correctly on a mid-panel month — the modeling pipeline here is the
same shape, run on the smaller public release instead.

**What's excluded, and why:**
- `trend_direction`, `trend_pct` — these *define* `is_declining_label`; using them as features
  would be reading the answer key (confirmed directly in `w06` Section 3: adding `trend_pct`
  pushes AUC from 0.604 to 0.998).
- `client_id`, `content_id` — pseudonyms, used only for the grouped train/test split, never as
  model inputs.
- FlyRank product decision flags (`health_score`, `priority_score`, `action_type`) — not shipped
  in this dataset at all, by design, so the model discovers its own signal instead of copying an
  existing one (per the lane guide's "observable signals, not product decisions" rule).
- Every row already satisfies `impressions_90d > 0` and `content_age_days >= 90` (the starter
  CSV's own construction rule) — brand-new, zero-visibility pages are out of scope for this
  playbook entirely.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print("Distinct clients:", df["client_id"].nunique())
print("Trailing window: 90 days ending at export time (one snapshot, not a time series)")


Rows, columns: (30000, 44)
Distinct clients: 32
Trailing window: 90 days ending at export time (one snapshot, not a time series)


## 3. Methodology

**Label:** `is_declining_label = (trend_direction == "down")` — a **proxy label**, computed from
a 30-day-vs-prior-30-day snapshot comparison, not an observed future outcome. Treated as such
throughout: every claim built on it uses observed/directional/decision-support language, never
causal language.

**Baseline (`w04`):** a transparent, hand-written rule — visible pages (≥500 impressions,
position 1–20) with CTR below 0.5% get scored by `log1p(impressions_90d) × (0.5 − ctr)`, one
reason code, one action label. Built on two signal checks: staleness behind FlyRank's refresh
flags (**MIXED** — decline rate reverses at the 181+ day bucket the flag targets) and CTR-vs-
position behind the CTR-fix logic (**CONFIRMED** — CTR falls monotonically as position worsens).

**Model (`w05`):** Logistic Regression and Random Forest, same features (18 numeric + 8
categorical + 5 missingness flags, all knowable at prediction time, none label-derived or a
product flag), same client-grouped 70/30 split, same precision@K metric as the baseline.

**Validation design (`w06`):** client-grouped split, chosen because content items from the same
client share hidden character that a random split would let the model memorize. Quantified: a
naive random split reports AUC 0.694; the honest grouped split reports AUC 0.604 — a real 0.09
gap attributable to memorization, not skill. The grouped number is what's reported everywhere
else in this capstone.

**Leakage checks (`w06`):** the full attack checklist — label-derived columns excluded and
confirmed absent, product flags confirmed absent from the dataset, population selection checked
for outcome-window information, split grouped by the repeating entity, base rate printed next to
every metric, and the deliberate-leak confession test (adding `trend_pct` back collapses the
problem to reading the answer key, AUC 0.604 → 0.998).

## 4. Results (vs baseline)

Same client-grouped holdout, same metric, as required by the training-honest-models skill — one
table, base rate included.

In [2]:
results_table = pd.DataFrame([
    {"K": 50,  "base_rate": 0.559, "baseline_rule": 0.620, "logistic_regression": 0.760, "random_forest": 0.540},
    {"K": 100, "base_rate": 0.559, "baseline_rule": 0.670, "logistic_regression": 0.690, "random_forest": 0.570},
    {"K": 250, "base_rate": 0.559, "baseline_rule": 0.676, "logistic_regression": 0.712, "random_forest": 0.628},
])
print(results_table.to_string(index=False))
print()
print("AUC -- logistic_regression: 0.604 | random_forest: 0.615 (client-grouped holdout)")
print("Random-split AUC for comparison (NOT used for any headline claim): 0.694")


  K  base_rate  baseline_rule  logistic_regression  random_forest
 50      0.559          0.620                0.760          0.540
100      0.559          0.670                0.690          0.570
250      0.559          0.676                0.712          0.628

AUC -- logistic_regression: 0.604 | random_forest: 0.615 (client-grouped holdout)
Random-split AUC for comparison (NOT used for any headline claim): 0.694


**Honest reading:** logistic regression beats the baseline at every K tested, by a modest
margin (biggest gap at K=50: 0.76 vs 0.62). Random forest is the more interesting result — it
underperforms the simple rule at K=50 and only pulls ahead at K=250, which is why logistic
regression, not the more complex model, carries forward into the action playbook (`w07`).

## 5. Limitations

- **Proxy label, not a future outcome.** `is_declining_label` is a current-window snapshot
  classification, not "this page will decline next month" — the stronger version of this project
  (predicting a genuinely future window) is future work, not what's measured here.
- **Cross-sectional, non-experimental data.** No A/B test, no before/after design with a control
  group. Nothing here can claim that refreshing a page *causes* recovery — only that certain
  pages are, in this slice, more likely to be labeled declining.
- **One 30,000-row slice of a ~79M-row warehouse.** Every number is "observed in this slice,"
  never "true of FlyRank clients in general."
- **Client-grouped evaluation still doesn't guarantee performance on a genuinely new client** —
  it only proves the model isn't purely memorizing the 22 training clients; `w07` Section 2 states
  this limit directly to the intended user.
- **The label itself is noisy at the row level** — `w05` Section 4 shows individual "wrong" cases
  where the model's reasoning was plausible but the 30-day snapshot label disagreed, which is a
  property of the label's short window, not necessarily a model failure.
- **No claim about Google's ranking algorithm.** Only search-console-style outcomes (impressions,
  clicks, position) are observed — never the mechanism that produces them.

## 6. Ranked recommendations

The action playbook's output (`w07`), summarized — full detail and the no-go list live in
`w07_action_playbook.ipynb`.

| Reason code | Action | Observed decline rate | n |
|---|---|---|---|
| `model_decline_risk` | `priority_refresh` | 73.7% | 9,643 |
| `ctr_review_candidate` | `review_ctr_and_meta` | 53.4% | 5,343 |
| `visible_model_opportunity` | `light_touch_review` | 52.4% | 1,084 |
| `engagement_review_candidate` | `review_content_depth` | 47.0% | 2,458 |
| *(none fired)* | `monitor` | 39.9% | 11,472 |

Overall base rate: 54.2%. `model_decline_risk` carries the clearest lift; every reason code stays
above the "monitor" bucket's rate, which is the honest signal that the ranking is doing real
work, not just reordering noise.

**Intended use:** an editor's weekly triage aid — never an automated edit, and never valid on a
client outside this training portfolio without re-validation (`w07` Section 2).

## 7. Artifacts the paper embeds

Three charts, generated once and reused by the deployed page (`docs/assets/`), all built from
numbers already shown above and in `w04`/`w05`/`w06`/`w07`.

In [3]:
import os
print("Chart files this capstone produced for the paper:")
for f in sorted(os.listdir("../outputs/charts")):
    print(" -", f)
print()
print("These live at work/outputs/charts/ and are copied into docs/assets/ for the deployed page.")


Chart files this capstone produced for the paper:
 - ctr_by_position.png
 - decline_rate_by_reason_code.png
 - precision_at_k.png

These live at work/outputs/charts/ and are copied into docs/assets/ for the deployed page.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
  **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.